# 01 — Agent Trajectory Generation

Generate labelled trajectory datasets from three agent types:
- **TruePreservationAgent**: hard-coded survival objective
- **InstrumentalAgent**: reward maximiser with instrumental survival
- **RandomAgent**: uniform random baseline

Outputs a dataset of shape `(N, T, 7)` with labels.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from src.agent_simulator import (
    GridWorld, TruePreservationAgent, InstrumentalAgent, RandomAgent,
    generate_dataset,
)

## 1. Visualise the GridWorld

In [ ]:
env = GridWorld()
grid = np.zeros((env.size, env.size))
for c in env.safe_zones: grid[c] = 1
for c in env.reward_tiles: grid[c] = 2
for c in env.terminal_cells: grid[c] = -1

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(grid, cmap='RdYlGn', origin='lower')
ax.set_title('GridWorld: green=safe, yellow=reward, red=terminal')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 2. Generate single trajectories for inspection

In [ ]:
agents = [
    ('TruePreservation', TruePreservationAgent(seed=0)),
    ('Instrumental', InstrumentalAgent(seed=0)),
    ('Random', RandomAgent(seed=0)),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, agent) in zip(axes, agents):
    traj = agent.generate_trajectory(T=100)
    ax.plot(traj[:, 0], traj[:, 1], 'o-', markersize=2, alpha=0.6)
    ax.set_title(f'{name} trajectory')
    ax.set_xlim(-0.5, 9.5)
    ax.set_ylim(-0.5, 9.5)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 3. Generate full dataset

In [ ]:
trajectories, labels, label_names = generate_dataset(
    n_per_class=200, T=100, seed=42
)
print(f'Trajectories shape: {trajectories.shape}')
print(f'Labels shape:       {labels.shape}')
print(f'Label names:        {label_names}')
print(f'Class counts:       {np.bincount(labels)}')

## 4. Feature distributions by agent class

In [ ]:
feature_names = ['x', 'y', 'action', 'reward', 'safety_signal', 'goal_signal', 'alive']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for fi in range(7):
    ax = axes[fi]
    for li, ln in enumerate(label_names):
        vals = trajectories[labels == li, :, fi].ravel()
        ax.hist(vals, bins=30, alpha=0.5, label=ln, density=True)
    ax.set_title(feature_names[fi])
    ax.legend(fontsize=7)
axes[-1].axis('off')
plt.tight_layout()
plt.show()

## 5. Save dataset

In [ ]:
os.makedirs('../data', exist_ok=True)
np.savez(
    '../data/agent_trajectories.npz',
    trajectories=trajectories,
    labels=labels,
    label_names=label_names,
)
print('Saved to data/agent_trajectories.npz')